# DT401 Week 5 Section 1: Data Analysis Lifecycle

**Module:** Data Fundamentals  
**Week:** 5 of 7  
**Section:** 1 of 3  
**Estimated Time:** 45 minutes

---

## Learning Objective

Understand and apply the six-stage data analysis lifecycle to workplace projects.

By the end of this notebook, you will be able to:
- Identify the six stages of the data analysis lifecycle
- Explain the purpose and activities in each lifecycle stage
- Apply Achieve-Measure-Verify principles to project completion
- Plan stakeholder engagement for data analytics projects

---

## Context Recap

In Week 4, you explored data gathering methods and descriptive statistics to understand datasets. This week advances to the complete data analysis lifecycle—the structured framework that guides analytical projects from initial business question to operational deployment. Understanding this lifecycle ensures your technical work creates genuine business value.

**Weather data from Open-Meteo API (https://open-meteo.com) - London, UK (51.49°N, 0.16°W), Jan 2023-Jan 2026**

---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>

<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>

<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints help guide your thinking without giving away the full answer.

</details>

</div>

---

## Part 1: Understanding the Data Analysis Lifecycle

### Visual Overview

```mermaid
graph TD
    A[Discovery] --> B[Data Preparation]
    B --> C[Model Planning]
    C --> D[Model Building]
    D --> E[Communicate Insights]
    E --> F[Operationalise]
    F -.-> A
    
    style A fill:#e1f5ff
    style B fill:#e1f5ff
    style C fill:#fff4e1
    style D fill:#fff4e1
    style E fill:#e1ffe1
    style F fill:#e1ffe1
```

**What this diagram shows:**
The data analysis lifecycle consists of six sequential stages, from discovering the business problem through to operationalising the solution. The dotted line shows that insights from deployed models can reveal new business questions, starting another cycle.

**Why this matters in your role:**
Understanding the lifecycle ensures you don't jump straight to building models without understanding business requirements. It helps you position your analytical work appropriately and communicate progress to stakeholders.

---

### The Six Lifecycle Stages

Let's explore each stage using a weather forecasting scenario to demonstrate lifecycle application. We'll use real weather data to show how each stage would work in practice.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load weather dataset
weather_data = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, low_memory=False)
weather_data.columns = [c.split(' ')[0].strip() for c in weather_data.columns]
weather_data = weather_data.rename(columns={'time': 'date'})

# Keep only hourly rows (YYYY-MM-DDTHH:MM format)
# The CSV contains a second daily-data section appended after the hourly data
weather_data = weather_data[weather_data['date'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)

# Convert all measurement columns to numeric
for col in weather_data.columns[1:]:
    weather_data[col] = pd.to_numeric(weather_data[col], errors='coerce')
weather_data['wind_speed_10m'] = weather_data['wind_speed_10m'] * 1.60934

print(f"Dataset loaded: {len(weather_data):,} hourly observations")
weather_data.head()

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
In real projects, the discovery stage happens BEFORE you start loading data. We're loading data here to demonstrate lifecycle stages, but in practice, discovery would identify which data you need.
</div>

### Stage 1: Discovery

**Purpose:** Define the business problem and establish success criteria.

**Example Business Question:**
"An outdoor events company wants to reduce event cancellations due to unexpected bad weather. Can we predict temperature 24 hours ahead with 80% accuracy?"

**Discovery Activities:**
- Define business problem: Reduce weather-related cancellations
- Identify stakeholders: Event managers, operations team, customers
- Establish success metrics: 80% prediction accuracy, 24-hour lead time
- Determine data sources: Historical weather observations, forecasts

In [ ]:
# Discovery: Understand data availability
print("Data Discovery Summary:")
print(f"Time period: {weather_data['date'].min()} to {weather_data['date'].max()}")
print(f"Total observations: {len(weather_data):,}")
print(f"\nAvailable measurements: {weather_data.shape[1]} variables")
print(weather_data.columns.tolist()[:10])  # Show first 10 columns

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We confirmed data availability covering 3 years (2023-2026)
- We identified multiple weather variables available for modeling
- This discovery step confirms we have sufficient data for the business question

**Key lesson:** Discovery validates feasibility before investing significant effort.

</div>

### Stage 2: Data Preparation

**Purpose:** Gather, clean, and transform data into analysis-ready format.

**Activities:**
- Extract relevant variables
- Handle missing values
- Create derived features
- Validate data quality

In [ ]:
# Data Preparation: Extract and clean temperature data
temp_data = weather_data[['date', 'temperature_2m']].copy()
temp_data['date'] = pd.to_datetime(temp_data['date'])

# Check for missing values
missing_pct = (temp_data['temperature_2m'].isna().sum() / len(temp_data)) * 100
print(f"Missing temperature values: {missing_pct:.2f}%")

# Remove missing values
temp_data_clean = temp_data.dropna()
print(f"Clean dataset: {len(temp_data_clean):,} observations ready for analysis")

### Stage 3: Model Planning

**Purpose:** Determine analytical approach based on business question.

**Decisions:**
- Model type: Time-series regression (predict continuous value)
- Features: Previous 24 hours temperature, hour of day, season
- Success criteria: 80% accuracy (predictions within ±2°C)

In [ ]:
# Model Planning: Visualize temperature patterns
# Sample one week to see patterns
sample_week = temp_data_clean[temp_data_clean['date'].dt.year == 2024].iloc[:168]  # 7 days * 24 hours

plt.figure(figsize=(12, 4))
plt.plot(sample_week['date'], sample_week['temperature_2m'])
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.title('Temperature Patterns (One Week Sample)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Planning insight: Temperature shows daily cycles - suitable for time-series modeling")

### Stage 4: Model Building

**Purpose:** Develop and train predictive models.

**Note:** Full model building will be covered in Section 2 of this week. Here we demonstrate the concept.

In [ ]:
# Model Building: Simple demonstration (not full model)
# Calculate 24-hour temperature persistence (naive forecast)
temp_data_clean['temp_24h_ago'] = temp_data_clean['temperature_2m'].shift(24)
temp_data_clean['prediction_error'] = abs(temp_data_clean['temperature_2m'] - temp_data_clean['temp_24h_ago'])

# Calculate accuracy (within ±2°C)
accurate_predictions = (temp_data_clean['prediction_error'] <= 2).sum()
total_predictions = temp_data_clean['prediction_error'].notna().sum()
accuracy = (accurate_predictions / total_predictions) * 100

print(f"Baseline model accuracy: {accuracy:.1f}%")
print(f"Target accuracy: 80%")
print(f"Status: {'✓ Target met' if accuracy >= 80 else '✗ Need improvement'}")

### Stage 5: Communicate Insights

**Purpose:** Translate technical results into business language.

**Key Message:**
"Our temperature forecasting model achieves 85% accuracy 24 hours ahead (exceeding 80% target). This enables event managers to make confident cancellation decisions with sufficient notice, reducing both unexpected cancellations and customer disappointment."

In [ ]:
# Communicate: Visualize model performance
# Sample predictions vs actuals
sample = temp_data_clean.dropna().iloc[1000:1168]  # One week sample

plt.figure(figsize=(12, 4))
plt.plot(range(len(sample)), sample['temperature_2m'], label='Actual', alpha=0.7)
plt.plot(range(len(sample)), sample['temp_24h_ago'], label='24h Forecast', alpha=0.7, linestyle='--')
plt.xlabel('Hours')
plt.ylabel('Temperature (°C)')
plt.title('Forecast Performance Example (One Week)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Business insight: Forecasts closely track actual temperatures, providing reliable decision support.")

### Stage 6: Operationalise

**Purpose:** Deploy model into production for ongoing use.

**Implementation:**
- Automate daily forecast generation
- Integrate with event management system
- Monitor model performance over time
- Plan retraining schedule

**Business Value:**
- Reduce weather-related cancellations by 40%
- Improve customer satisfaction scores
- Optimize resource allocation for events

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

We demonstrated all six lifecycle stages using weather forecasting:
- **Discovery:** Defined business problem and success criteria
- **Data Preparation:** Cleaned and validated temperature data
- **Model Planning:** Chose time-series approach based on patterns
- **Model Building:** Developed baseline forecasting model
- **Communicate Insights:** Visualized performance for stakeholders
- **Operationalise:** Described deployment and business value

**Key lesson:** The lifecycle connects technical work to business outcomes at every stage.

</div>

---

## Part 2: Achieve-Measure-Verify Framework

### Visual Overview

```mermaid
graph LR
    A[Achieve:<br/>Deliver Solution] --> B[Measure:<br/>Calculate Success Metrics]
    B --> C[Verify:<br/>Confirm with Stakeholders]
    C --> D[Plan Next Steps]
    
    style A fill:#e1f5ff
    style B fill:#fff4e1
    style C fill:#e1ffe1
    style D fill:#ffe1e1
```

**What this diagram shows:**
Project completion follows three steps: achieve your objectives, measure success quantitatively, and verify results with stakeholders before planning next steps.

**Why this matters in your role:**
This framework ensures you don't just deliver technical outputs—you demonstrate business value and maintain stakeholder trust through evidence-based reporting.

---

### Worked Example: Project Completion

Let's demonstrate the Achieve-Measure-Verify framework using our weather forecasting project.

In [ ]:
# ACHIEVE: Calculate comprehensive success metrics
results = temp_data_clean.dropna()

# Metric 1: Accuracy (within ±2°C)
within_2deg = (results['prediction_error'] <= 2).sum()
accuracy = (within_2deg / len(results)) * 100

# Metric 2: Average error
avg_error = results['prediction_error'].mean()

# Metric 3: Worst-case error (95th percentile)
worst_case = results['prediction_error'].quantile(0.95)

print("=== ACHIEVE: Project Completed ===")
print(f"✓ Forecasting model deployed")
print(f"✓ Integration with event system complete")
print(f"✓ Monitoring dashboard active")

In [ ]:
# MEASURE: Report quantitative success
print("\n=== MEASURE: Success Metrics ===")
print(f"Target accuracy: 80%")
print(f"Achieved accuracy: {accuracy:.1f}%")
print(f"Status: {'✓ EXCEEDED' if accuracy > 80 else '✓ MET' if accuracy >= 80 else '✗ BELOW TARGET'}")
print(f"\nAverage prediction error: {avg_error:.2f}°C")
print(f"95th percentile error: {worst_case:.2f}°C")

In [ ]:
# VERIFY: Stakeholder confirmation template
print("\n=== VERIFY: Stakeholder Report ===")
print("\nTo: Event Operations Manager")
print("Subject: Weather Forecasting Project - Success Confirmation\n")
print(f"Our forecasting model achieves {accuracy:.1f}% accuracy, exceeding the 80% target.")
print(f"This enables confident event decisions 24 hours ahead.\n")
print("Estimated business impact:")
print("- 40% reduction in last-minute cancellations")
print("- Improved customer satisfaction")
print("- Better resource planning\n")
print("Next steps: Monitor performance, plan quarterly model retraining.")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- **Achieve:** Confirmed all project deliverables completed
- **Measure:** Calculated specific success metrics against targets
- **Verify:** Prepared stakeholder-ready summary with business impact

**Key lesson:** Quantitative metrics build stakeholder trust and justify continued investment in analytics.

</div>

---

### Modified Task: Your Project Success Metrics

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Think about a recent project in your workplace (doesn't need to be data-related). How would you apply Achieve-Measure-Verify?

Complete the template below:

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Think about:
- ACHIEVE: What were the deliverables? Were they completed?
- MEASURE: What quantitative metrics showed success? (time saved, cost reduced, quality improved)
- VERIFY: How did you confirm success with stakeholders?

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for an example</summary>

**Project:** Implementing new customer database system

**ACHIEVE:**
- ✓ Database system deployed
- ✓ Staff training completed
- ✓ Data migration successful

**MEASURE:**
- Target: Reduce customer lookup time by 50%
- Achieved: 60% reduction (2 minutes → 48 seconds)
- Database uptime: 99.8% (exceeded 99% target)

**VERIFY:**
- Customer service team reports faster response times
- Customer satisfaction scores increased 12%
- Manager confirmed project success in monthly review

</details>

</div>

In [ ]:
# Your reflection (write in code cell for portfolio)
your_project = {
    'project_name': '',  # What project?
    'achieve': [],  # List deliverables completed
    'measure': {},  # Dict of metrics: {metric_name: value}
    'verify': ''  # How did you confirm success?
}

# Example:
# your_project = {
#     'project_name': 'Customer Database Implementation',
#     'achieve': ['Database deployed', 'Training completed', 'Data migrated'],
#     'measure': {'lookup_time_reduction': '60%', 'uptime': '99.8%'},
#     'verify': 'Manager confirmed success in monthly review'
# }

print("Project Reflection Template:")
print(f"Project: {your_project['project_name']}")
print(f"\nACHIEVE: {your_project['achieve']}")
print(f"MEASURE: {your_project['measure']}")
print(f"VERIFY: {your_project['verify']}")

**Expected outcome:** Completed reflection on how Achieve-Measure-Verify applies to your workplace projects.

**Success check:** You've identified quantitative metrics and stakeholder verification for a real project.

---

## Part 3: Stakeholder Engagement Planning

### Visual Overview

```mermaid
graph TD
    A[Identify Stakeholders] --> B[Map Interest/Influence]
    B --> C[Plan Communication]
    C --> D[Present Findings]
    
    B --> E[High Interest<br/>High Influence]
    B --> F[High Interest<br/>Low Influence]
    B --> G[Low Interest<br/>High Influence]
    B --> H[Low Interest<br/>Low Influence]
    
    E --> I[Engage Closely]
    F --> J[Keep Informed]
    G --> K[Keep Satisfied]
    H --> L[Monitor]
    
    style E fill:#ff9999
    style F fill:#ffcc99
    style G fill:#ffcc99
    style H fill:#ccffcc
```

**What this diagram shows:**
Stakeholders vary in interest and influence. Your engagement strategy should match their position: closely engage high-interest/high-influence stakeholders, keep others appropriately informed.

**Why this matters in your role:**
Effective stakeholder management prevents project delays and ensures your analytical work addresses the right business questions.

---

### Worked Example: Stakeholder Mapping

Let's map stakeholders for a healthcare patient flow optimization project.

In [ ]:
# Define stakeholders for patient flow project
stakeholders = [
    {'name': 'Clinical Director', 'interest': 9, 'influence': 9, 'group': 'Internal'},
    {'name': 'Ward Managers', 'interest': 10, 'influence': 6, 'group': 'Internal'},
    {'name': 'IT Department', 'interest': 4, 'influence': 7, 'group': 'Internal'},
    {'name': 'Finance Team', 'interest': 7, 'influence': 8, 'group': 'Internal'},
    {'name': 'Nursing Staff', 'interest': 8, 'influence': 5, 'group': 'Internal'},
    {'name': 'Patients', 'interest': 10, 'influence': 3, 'group': 'External'}
]

# Visualize stakeholder map
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
for s in stakeholders:
    color = 'blue' if s['group'] == 'Internal' else 'green'
    plt.scatter(s['influence'], s['interest'], s=500, alpha=0.6, c=color)
    plt.text(s['influence'], s['interest'], s['name'], 
             ha='center', va='center', fontsize=9, fontweight='bold')

plt.axhline(y=7, color='red', linestyle='--', alpha=0.3)
plt.axvline(x=7, color='red', linestyle='--', alpha=0.3)
plt.xlabel('Influence (1-10)', fontsize=12)
plt.ylabel('Interest (1-10)', fontsize=12)
plt.title('Stakeholder Mapping: Patient Flow Optimization', fontsize=14)
plt.xlim(0, 11)
plt.ylim(0, 11)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Categorize stakeholders by engagement strategy
for s in stakeholders:
    if s['interest'] >= 7 and s['influence'] >= 7:
        strategy = "ENGAGE CLOSELY - Regular updates, involve in decisions"
    elif s['interest'] >= 7 and s['influence'] < 7:
        strategy = "KEEP INFORMED - Regular updates, gather feedback"
    elif s['interest'] < 7 and s['influence'] >= 7:
        strategy = "KEEP SATISFIED - Key milestone updates, address concerns"
    else:
        strategy = "MONITOR - Occasional updates, available if needed"
    
    print(f"{s['name']:20} | Interest: {s['interest']}/10 | Influence: {s['influence']}/10 | {strategy}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We identified 6 stakeholders for a patient flow project
- We scored each by interest (how much they care) and influence (their decision-making power)
- We visualized positions to identify engagement priorities
- We determined appropriate engagement strategies for each

**Key lesson:** Not all stakeholders need the same engagement level. Prioritize based on interest and influence to use your time effectively.

</div>

---

## Part 4: Independent Task - Lifecycle Planning for Your Workplace

### Your Challenge

You work in a retail organization noticing increasing customer complaints about product recommendations being irrelevant. Your manager asks you to explore whether data analysis could improve recommendation quality.

**Plan a data analysis project using the lifecycle framework:**

1. **Discovery:** Define the business problem, success metrics, and data sources needed
2. **Data Preparation:** List what data preparation steps would be needed
3. **Model Planning:** What type of model would you choose and why?
4. **Model Building:** What would success look like quantitatively?
5. **Communicate Insights:** How would you present findings to management?
6. **Operationalise:** How would the solution be deployed?

**Stakeholder Mapping:**
- Identify 5 stakeholders for this project
- Rate their interest (1-10) and influence (1-10)
- Determine appropriate engagement strategy for each

**Getting started:**
- Think about real customer complaints you've seen
- Consider what data might already exist (purchase history, browsing data, customer demographics)
- Remember: supervised models need labeled examples (good vs bad recommendations)

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Complete the lifecycle planning template below for the retail recommendation project.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

**Discovery hints:**
- Business problem: "Improve recommendation relevance to reduce complaints"
- Success metric: "Reduce complaints by 40%" or "Increase click-through on recommendations by 30%"
- Data sources: Customer purchase history, product catalog, browsing behavior

**Model Planning hint:**
- This is a recommendation problem - could use clustering (group similar customers) or classification (predict what customer will like)

**Stakeholders hint:**
- Consider: Customer Service Manager, Marketing Director, IT Development Team, Store Managers, Customers

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Lifecycle Plan
project_plan = {
    'discovery': {
        'business_problem': 'Improve product recommendation relevance to reduce customer complaints',
        'success_metrics': ['Reduce complaints by 40%', 'Increase recommendation click-through by 30%'],
        'data_sources': ['Purchase history database', 'Web analytics', 'Product catalog', 'Customer demographics']
    },
    'data_preparation': [
        'Extract 2 years purchase history',
        'Clean: remove returns, cancelled orders',
        'Create features: customer segments, product categories, purchase frequency',
        'Label data: identify which recommendations led to purchases (positive) vs ignored (negative)'
    ],
    'model_planning': {
        'model_type': 'Supervised classification - predict likelihood customer will click/buy recommended product',
        'features': ['Customer segment', 'Past purchase categories', 'Browsing history', 'Time since last purchase'],
        'target': 'Click/purchase on recommendation (yes/no)'
    },
    'model_building': {
        'success_criteria': 'Model predicts successful recommendations with 75% accuracy',
        'validation': 'Test on most recent 3 months data (not used in training)'
    },
    'communicate_insights': {
        'key_message': 'New recommendation model increases relevance by 35%, projected to reduce complaints by 45%',
        'visualizations': ['Before/after recommendation relevance scores', 'Customer segment response rates']
    },
    'operationalise': {
        'deployment': 'Integrate model with website recommendation engine',
        'monitoring': 'Track daily: click-through rate, complaints, conversion rate',
        'maintenance': 'Retrain model monthly with new purchase data'
    }
}

# Stakeholder Mapping
stakeholders = [
    {'name': 'Customer Service Manager', 'interest': 10, 'influence': 7, 'strategy': 'ENGAGE CLOSELY'},
    {'name': 'Marketing Director', 'interest': 9, 'influence': 9, 'strategy': 'ENGAGE CLOSELY'},
    {'name': 'IT Development Team', 'interest': 5, 'influence': 8, 'strategy': 'KEEP SATISFIED'},
    {'name': 'Store Managers', 'interest': 8, 'influence': 5, 'strategy': 'KEEP INFORMED'},
    {'name': 'Customers', 'interest': 10, 'influence': 4, 'strategy': 'KEEP INFORMED'}
]
```

</details>

</div>

In [ ]:
# Your lifecycle plan (write in code cell for portfolio)
project_plan = {
    'discovery': {
        'business_problem': '',
        'success_metrics': [],
        'data_sources': []
    },
    'data_preparation': [],
    'model_planning': {
        'model_type': '',
        'features': [],
        'target': ''
    },
    'model_building': {
        'success_criteria': '',
        'validation': ''
    },
    'communicate_insights': {
        'key_message': '',
        'visualizations': []
    },
    'operationalise': {
        'deployment': '',
        'monitoring': '',
        'maintenance': ''
    }
}

# Your stakeholder mapping
stakeholders = [
    {'name': '', 'interest': 0, 'influence': 0, 'strategy': ''},
    # Add 4 more stakeholders
]

print("=== LIFECYCLE PLAN ===")
print(f"Discovery: {project_plan['discovery']}")
print(f"\nData Preparation: {project_plan['data_preparation']}")
print(f"\nModel Planning: {project_plan['model_planning']}")
print(f"\nModel Building: {project_plan['model_building']}")
print(f"\nCommunicate: {project_plan['communicate_insights']}")
print(f"\nOperationalise: {project_plan['operationalise']}")
print(f"\n=== STAKEHOLDERS ===")
for s in stakeholders:
    if s['name']:  # Only print if stakeholder added
        print(f"{s['name']}: Interest={s['interest']}, Influence={s['influence']}, Strategy={s['strategy']}")

**Success criteria:**
- Complete lifecycle plan covering all 6 stages
- Specific, measurable success metrics defined
- Appropriate model type selected with justification
- 5 stakeholders mapped with engagement strategies
- Business context maintained throughout (not just technical details)

**Portfolio note:** This lifecycle planning demonstrates your ability to structure data analysis projects (MLO 3: Select and apply appropriate data-gathering methods). Include this in your Module Challenge Part A portfolio.

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against:
- **MLO1:** *Explain the roles and functions of technology solutions within an organisation*
- **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Identify a project in your organisation where multiple roles, such as project managers, analysts, developers and business stakeholders, needed to collaborate. Using the interest/influence matrix from this notebook, map these roles and stakeholders, and explain how better collaboration between them could have improved the project's outcome.

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 5 Section 1.** You can now:

- Explored the six-stage data analysis lifecycle (Discovery, Data Preparation, Model Planning, Model Building, Communicate Insights, Operationalise) using the Open-Meteo London weather dataset to build a 24-hour temperature persistence forecast
- Applied the Achieve-Measure-Verify framework to the weather forecasting project, calculating accuracy, average error, and 95th-percentile error against a target
- Mapped six stakeholders for a patient flow optimisation project on an interest/influence matrix and assigned engagement strategies
- Planned a complete lifecycle project, from discovery to operationalisation, for a retail recommendation-quality problem in your own workplace

**Key takeaways:**

- The lifecycle is cyclic, not linear: insights from Operationalise can surface new business questions that restart Discovery
- Stakeholder engagement strategy is set by interest and influence, not organisational seniority alone: a Ward Manager (high interest, moderate influence) needs different handling from an IT Department (low interest, high influence)

**What comes next:**

- **Section 2a: Supervised Model, Linear Regression** builds on Stage 4 (Model Building) of the lifecycle, using linear regression to build a working predictive model from the weather dataset introduced here.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 5 Section 1 Data Analysis Lifecycle"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.5 component on Aptem
5. Open `DT401_Week5_Section2a.ipynb` when ready to continue